# 06 — Recherche par similarité

Choisis une image, le nombre de résultats et le poids de la couleur. Une distance faible indique une signature proche. Le notebook 05 génère l’index requis.

**Préparer le noyau pour la recherche**

- La commande d’installation est commentée. Si un import échoue, retire le caractère `#` et exécute cette cellule dans le noyau du projet. Les paquets servent à lire l’index et la requête, calculer les descripteurs et afficher les résultats.
- La recherche a aussi besoin du fichier `data/index_caltech101.npz` généré au notebook 05. Ce fichier contient les signatures de la base déjà calculées.

In [ ]:
#%pip install pillow numpy matplotlib scikit-image

**Transformer la requête comme les images indexées**

- En recevant une image, `signatures` la convertit en RGB carré de 256 × 256 puis construit un tableau HSV et un tableau d’intensités. Le fichier source de la requête peut venir de Caltech-101 ou être un autre fichier lisible par Pillow.
- La fonction utilise les mêmes intervalles HSV, le même LBP et les mêmes conventions de normalisation que l’indexeur. Ainsi, la case de rang `i` d’un histogramme de requête a la même signification que la case `i` de chaque ligne de l’index.
- Seule l’image de requête passe par cette extraction. Les descripteurs des candidats sont chargés depuis l’archive.

In [ ]:
from pathlib import Path
from time import perf_counter
from uuid import uuid4
import numpy as np
from PIL import Image
from skimage.feature import local_binary_pattern

def signatures(chemin):
    with Image.open(chemin) as image:
        rgb = image.convert("RGB").resize((256, 256), Image.Resampling.LANCZOS)
    hsv = np.asarray(rgb.convert("HSV"))
    gris = np.asarray(rgb.convert("L"))
    couleur, _ = np.histogramdd(hsv.reshape(-1, 3), bins=(8, 8, 8),
                               range=((0, 256), (0, 256), (0, 256)))
    lbp = local_binary_pattern(gris, 8, 1, method="uniform")
    texture, _ = np.histogram(lbp, bins=np.arange(11))
    return (couleur / couleur.sum()).ravel(), texture / texture.sum()

**Calculer les distances, exclure la requête et afficher le top N**

- `nombre` règle la taille du classement et `poids_couleur` règle le mélange HSV/LBP. Après chargement de l’archive, NumPy multiplie la signature par chaque ligne de la matrice en une opération. La somme sur `axis=1` donne un coefficient de recouvrement par image. Le test sur la matrice identité vérifie une distance de 0 pour deux histogrammes identiques et de 1 pour deux histogrammes disjoints.
- Si la requête est sous le dossier de la base, le code calcule son chemin relatif, puis le retire des candidats au moyen d’une comparaison insensible à la casse. La requête reste visible dans la galerie comme point de référence. `argsort` trie les indices selon les scores. Le tri stable garde l’ordre de l’index en cas d’égalité.
- Les résultats sont affichés avec leur catégorie et leur distance. Tu peux comparer les couleurs ou les textures visibles pour comprendre certains rapprochements. Le chronomètre s’arrête avant la galerie. Il inclut le chargement, l’extraction de la requête et le classement, tandis que l’ouverture des résultats pour l’affichage vient ensuite.

In [ ]:
import matplotlib.pyplot as plt

base = Path("data/101_ObjectCategories")
requete = base / "airplanes/image_0001.jpg"  # Remplace ce chemin pour tester une autre image.
nombre = 5
poids_couleur = 0.5
if nombre < 1 or not 0 <= poids_couleur <= 1:
    raise ValueError("Choisir N >= 1 et un poids entre 0 et 1.")

debut = perf_counter()
with np.load("data/index_caltech101.npz", allow_pickle=False) as index:
    chemins = index["chemins"]
    categories = index["categories"]
    couleurs = index["couleurs"]
    textures = index["textures"]

def distances(histogrammes, signature):
    coefficient = np.sqrt(histogrammes * signature).sum(axis=1)
    return np.sqrt(np.clip(1 - coefficient, 0, 1))

assert np.allclose(distances(np.eye(2), np.array([1, 0])), [0, 1])
couleur, texture = signatures(requete)
scores = poids_couleur * distances(couleurs, couleur) + (1 - poids_couleur) * distances(textures, texture)
if requete.resolve().is_relative_to(base.resolve()):
    relative = requete.resolve().relative_to(base.resolve()).as_posix()
    candidats = np.flatnonzero(np.char.lower(chemins) != relative.lower())
else:
    candidats = np.arange(len(chemins))
classement = candidats[np.argsort(scores[candidats], kind="stable")][:nombre]
print(f"Recherche sur CPU (chargement inclus) : {perf_counter() - debut:.3f} s")

fig, axes = plt.subplots(1, len(classement) + 1, figsize=(3 * (len(classement) + 1), 3), squeeze=False)
resultats = [(requete, "Requête")]
resultats += [(base / chemins[i], f"{rang}. {categories[i]}\nDistance : {scores[i]:.3f}")
              for rang, i in enumerate(classement, 1)]
for ax, (chemin, titre) in zip(axes[0], resultats):
    with Image.open(chemin) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(titre)
    ax.axis("off")
fig.tight_layout()
plt.show()